
# Greenhouse Data Cleaning Notebook

This notebook cleans the greenhouse workbook and exports a single analysis-ready CSV file.

## What changed in this upgraded version

This version is stricter and more structured than the first pass:

- it parses nutrient entries more accurately, including compact values like `4L 400A`
- it converts more planting notes into numeric plant counts
- it standardizes leak locations, crop labels, and problem categories
- it creates a unified `water_use_l` field for system comparison
- it converts obvious `didn't water` rows to `0` instead of leaving them ambiguous
- it drops truly irrelevant sparse rows rather than keeping every row by default

## Cleaning principles

- never invent critical measured values
- standardize anything that can be made cleaner without changing meaning
- keep meaningful event rows even if they do not contain a water measurement
- remove only the rows that are too sparse to support analysis


In [6]:

import re
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

# Upload file in Google Colab
from google.colab import files
uploaded = files.upload()

DATA_FILE = Path(list(uploaded.keys())[0])
OUTPUT_FILE = Path("greenhouse_systems_cleaned.csv")

NS = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
EXCEL_EPOCH = datetime(1899, 12, 30)

Saving Water consumption in glasshouse new cycle - original data.xlsx to Water consumption in glasshouse new cycle - original data (1).xlsx



## Step 1. Load the workbook without `openpyxl`

`openpyxl` is not available in this environment, so the notebook reads the `.xlsx` file directly as an OOXML zip archive.

Why this is a good approach:

- no dependency installation is needed
- source row numbers are preserved for traceability
- all visible sheets can still be loaded into one working table


In [7]:

def clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).replace("\u2019", "'").replace("\u2018", "'")
    text = re.sub(r"\s+", " ", text).strip()
    if text in {"", "-", "nan", "None"}:
        return pd.NA
    return text


def load_excel_xml(path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(path) as zf:
        shared_strings = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in root.findall("a:si", NS):
                shared_strings.append("".join((t.text or "") for t in si.findall(".//a:t", NS)))

        workbook = ET.fromstring(zf.read("xl/workbook.xml"))
        rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
        rel_map = {}
        for rel in rels:
            rid = rel.attrib.get("Id") or rel.attrib.get(
                "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}Id"
            )
            rel_map[rid] = "xl/" + rel.attrib["Target"].lstrip("/")

        def col_idx(cell_ref: str) -> int:
            letters = re.match(r"([A-Z]+)", cell_ref).group(1)
            idx = 0
            for ch in letters:
                idx = idx * 26 + (ord(ch) - 64)
            return idx - 1

        def cell_value(cell) -> str:
            cell_type = cell.attrib.get("t")
            value_elem = cell.find("a:v", NS)
            if cell_type == "inlineStr":
                inline = cell.find("a:is", NS)
                return "".join((t.text or "") for t in inline.findall(".//a:t", NS)) if inline is not None else ""
            if value_elem is None:
                return ""
            raw = value_elem.text or ""
            if cell_type == "s":
                try:
                    return shared_strings[int(raw)]
                except Exception:
                    return raw
            return raw

        rows_out = []
        for sheet in workbook.findall("a:sheets/a:sheet", NS):
            sheet_name = sheet.attrib["name"]
            target = rel_map[sheet.attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]]
            sheet_root = ET.fromstring(zf.read(target))
            rows = sheet_root.findall(".//a:sheetData/a:row", NS)

            parsed_rows = []
            max_width = 0
            for row in rows:
                cells = row.findall("a:c", NS)
                if not cells:
                    continue
                width = max(col_idx(c.attrib["r"]) for c in cells) + 1
                max_width = max(max_width, width)
                values = [""] * width
                for cell in cells:
                    values[col_idx(cell.attrib["r"])] = cell_value(cell).strip()
                parsed_rows.append(values)

            parsed_rows = [r + [""] * (max_width - len(r)) for r in parsed_rows]
            header = parsed_rows[0]
            for row_number, row in enumerate(parsed_rows[1:], start=2):
                if any(value not in {"", "-"} for value in row):
                    record = {"source_sheet": sheet_name, "source_row_number": row_number}
                    record.update(dict(zip(header, row)))
                    rows_out.append(record)

    return pd.DataFrame(rows_out)


raw_df = load_excel_xml(DATA_FILE)
for column in raw_df.columns:
    if column != "source_row_number":
        raw_df[column] = raw_df[column].apply(clean_text)

print("Raw imported rows:", len(raw_df))
print("Raw rows by sheet:")
print(raw_df["source_sheet"].value_counts())
raw_df.head()


Raw imported rows: 276
Raw rows by sheet:
source_sheet
Conventional           117
A shape and Gutters     97
Towers                  62
Name: count, dtype: int64


,source_sheet,source_row_number,Date,Time,water in return,mins added,return now,nutrient solution,pH down,how much consumed,leak or no,how many planted,problems?,plant,seed or seedling,age,watered amount,how many plants planted?,problems faced
0,Towers,2,45876,0.64930555555555558,<NA>,<NA>,160L,2L A 2L B,15ml,<NA>,<NA>,all except 1 unit,rotation and lights off when I arrive,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,Towers,3,45877,0.47152777777777777,130L,<NA>,<NA>,peristaltic pumps,15ml,30L,yes R1,all except 1 unit,rotation and lights off when I arrive,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,Towers,4,45877,0.55555555555555558,126L,3 mins,200L,peristaltic pumps,10ml,4L in 2 hours?,yes R1,all except 1 unit,rotation and lights off when I arrive,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,Towers,5,45880,0.49444444444444446,60L,3 mins,135L,peristaltic pumps,10ml,140L (weekend),yes R1,all except 1 unit,rotation and lights off when I arrive,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,Towers,6,45881,0.59166666666666667,90L,manually,160L,peristaltic pumps,15ml,45L,yes R1,all except 1 unit,electricity kept turning on and off,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>



## Step 2. Define parsing and standardization helpers

The workbook stores many values in free text rather than clean numeric columns.

These helpers handle:

- Excel serial dates and times
- liters and milliliters
- nutrient A/B parsing from compact strings
- refill durations
- leak flags, severity, and location
- growth stage and age parsing
- plant counts and crop labels
- issue categories and record-quality flags


In [8]:

def excel_serial_to_date(value):
    value = clean_text(value)
    if pd.isna(value):
        return pd.NaT
    try:
        return (EXCEL_EPOCH + timedelta(days=float(value))).date()
    except Exception:
        return pd.NaT


def excel_fraction_to_time(value):
    value = clean_text(value)
    if pd.isna(value):
        return pd.NA
    try:
        total_seconds = round(float(value) * 24 * 60 * 60)
        hours = (total_seconds // 3600) % 24
        minutes = (total_seconds % 3600) // 60
        seconds = total_seconds % 60
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"
    except Exception:
        return pd.NA


def volume_to_liters(value, unit):
    value = float(value)
    return value if unit.lower() == "l" else value / 1000


def volume_to_ml(value, unit):
    value = float(value)
    return value * 1000 if unit.lower() == "l" else value


def clean_comment_text(text):
    text = clean_text(text)
    if pd.isna(text):
        return pd.NA
    text = re.sub(r"^[,;:\-\s]+", "", text)
    text = re.sub(r"[,;:\-\s]+$", "", text)
    while text.startswith("(") and text.endswith(")") and len(text) > 2:
        candidate = text[1:-1].strip()
        if not candidate:
            break
        text = candidate
    return clean_text(text)


def split_value_and_comment(text):
    text = clean_text(text)
    if pd.isna(text):
        return np.nan, pd.NA
    match = re.search(r"(\d+(?:\.\d+)?)\s*(l|ml)\b", text, flags=re.IGNORECASE)
    if not match:
        return np.nan, clean_comment_text(text)
    value_l = volume_to_liters(match.group(1), match.group(2))
    remainder = f"{text[:match.start()]} {text[match.end():]}".strip()
    return value_l, clean_comment_text(remainder)


def parse_primary_volume_liters(text):
    return split_value_and_comment(text)[0]


def parse_total_volume_liters(text):
    text = clean_text(text)
    if pd.isna(text):
        return np.nan
    matches = re.findall(r"(\d+(?:\.\d+)?)\s*(l|ml)\b", text, flags=re.IGNORECASE)
    if not matches:
        return np.nan
    total = 0.0
    for value, unit in matches:
        total += volume_to_liters(value, unit)
    return total


def parse_watered_amount_liters(text):
    text = clean_text(text)
    if pd.isna(text):
        return np.nan
    lowered = text.lower()
    if "didnt water" in lowered or "didn't water" in lowered:
        return 0.0
    return parse_total_volume_liters(text)


def normalize_nutrient_text(text):
    text = clean_text(text)
    if pd.isna(text):
        return pd.NA
    text = re.sub(r"([A-Za-z])(?=\d)", r"\1 ", text)
    text = re.sub(r"(?<=\d)([A-Za-z])", r" \1", text)
    text = re.sub(r"\bto\s+(nft|gutters?|towers?)\b", r" \1 ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_amount_fragment_ml(fragment: str):
    fragment = fragment.strip()
    if not fragment:
        return np.nan
    tokens = re.findall(r"(\d+(?:\.\d+)?)(?:\s*(l|ml))?", fragment, flags=re.IGNORECASE)
    if not tokens:
        return np.nan
    total_ml = 0.0
    for value, unit in tokens:
        total_ml += volume_to_ml(value, unit or "ml")
    return total_ml


def normalize_location_label(label):
    if not label:
        return None
    lowered = label.lower()
    if lowered.startswith("gutter"):
        return "gutters"
    if lowered.startswith("tower"):
        return "towers"
    return "nft"


def parse_nutrient_solution(text, system):
    text = normalize_nutrient_text(text)
    result = {
        "nutrient_a_ml": np.nan,
        "nutrient_b_ml": np.nan,
        "nutrient_total_ml": np.nan,
        "nutrient_solution_note": pd.NA,
        "peristaltic_pump": "No",
        "nutrient_a_nft_ml": np.nan,
        "nutrient_b_nft_ml": np.nan,
        "nutrient_a_gutters_ml": np.nan,
        "nutrient_b_gutters_ml": np.nan,
        "nutrient_a_towers_ml": np.nan,
        "nutrient_b_towers_ml": np.nan,
    }
    if pd.isna(text):
        return result

    lowered = text.lower()
    if "peristaltic pump" in lowered:
        result["peristaltic_pump"] = "Yes"
        return result
    if "drained all the water" in lowered:
        result["nutrient_solution_note"] = "System drained and replaced"
        return result

    pair_pattern = re.compile(
        r"((?:\d+(?:\.\d+)?\s*(?:l|ml)?\s*)+)A\s*((?:\d+(?:\.\d+)?\s*(?:l|ml)?\s*)+)B(?:\s*(NFT|gutters?|towers?))?",
        flags=re.IGNORECASE,
    )

    a_total = 0.0
    b_total = 0.0
    a_found = False
    b_found = False
    location_totals = {
        "nft": {"a": 0.0, "b": 0.0, "a_found": False, "b_found": False},
        "gutters": {"a": 0.0, "b": 0.0, "a_found": False, "b_found": False},
        "towers": {"a": 0.0, "b": 0.0, "a_found": False, "b_found": False},
    }

    for match in pair_pattern.finditer(text):
        a_fragment, b_fragment, label = match.groups()
        a_amount_ml = parse_amount_fragment_ml(a_fragment)
        b_amount_ml = parse_amount_fragment_ml(b_fragment)

        if not np.isnan(a_amount_ml):
            a_total += a_amount_ml
            a_found = True
        if not np.isnan(b_amount_ml):
            b_total += b_amount_ml
            b_found = True

        location = normalize_location_label(label)
        if not location and system == "Tower":
            location = "towers"
        if location:
            if not np.isnan(a_amount_ml):
                location_totals[location]["a"] += a_amount_ml
                location_totals[location]["a_found"] = True
            if not np.isnan(b_amount_ml):
                location_totals[location]["b"] += b_amount_ml
                location_totals[location]["b_found"] = True

    if a_found:
        result["nutrient_a_ml"] = a_total
    if b_found:
        result["nutrient_b_ml"] = b_total
    if a_found or b_found:
        result["nutrient_total_ml"] = np.nansum([result["nutrient_a_ml"], result["nutrient_b_ml"]])

    for location, values in location_totals.items():
        if values["a_found"]:
            result[f"nutrient_a_{location}_ml"] = values["a"]
        if values["b_found"]:
            result[f"nutrient_b_{location}_ml"] = values["b"]

    residual = pair_pattern.sub(" ", text)
    residual = re.sub(r"\band\b", " ", residual, flags=re.IGNORECASE)
    residual = clean_comment_text(residual)
    if not pd.isna(residual):
        result["nutrient_solution_note"] = residual
    return result


def classify_nutrient_entry(text):
    text = normalize_nutrient_text(text)
    if pd.isna(text):
        return "No Addition"
    lowered = text.lower()
    if "peristaltic pump" in lowered:
        return "Peristaltic pumps"
    if "drained all the water" in lowered:
        return "Drain and replace"
    if re.search(r"\d", lowered) and re.search(r"\b[ab]\b", lowered):
        return "Measured addition"
    return "No Addition"


def parse_duration_minutes(text):
    text = clean_text(text)
    if pd.isna(text):
        return np.nan, pd.NA, False
    lowered = text.lower()
    if "manual" in lowered:
        return np.nan, "Manual", True
    if "end of cycle" in lowered:
        return np.nan, "End of cycle", False

    minutes = 0.0
    found = False
    for value in re.findall(r"(\d+(?:\.\d+)?)\s*min", lowered):
        minutes += float(value)
        found = True
    for value in re.findall(r"(\d+(?:\.\d+)?)\s*sec", lowered):
        minutes += float(value) / 60
        found = True

    if found:
        return round(minutes, 3), "Timed", False
    return np.nan, pd.NA, False


def parse_ph_down_by_location(text, system):
    text = normalize_nutrient_text(text)
    result = {
        "ph_down_ml": np.nan,
        "ph_down_nft_ml": np.nan,
        "ph_down_gutters_ml": np.nan,
        "ph_down_towers_ml": np.nan,
    }
    if pd.isna(text):
        return result

    amount_pattern = re.compile(
        r"(\d+(?:\.\d+)?)\s*(ml|l)\b(?:\s*(NFT|gutters?|towers?))?",
        flags=re.IGNORECASE,
    )
    location_totals = {"nft": 0.0, "gutters": 0.0, "towers": 0.0}
    location_found = {"nft": False, "gutters": False, "towers": False}
    unlabeled_amounts = []
    total = 0.0
    found = False

    for match in amount_pattern.finditer(text):
        amount_ml = volume_to_ml(match.group(1), match.group(2))
        location = normalize_location_label(match.group(3))
        total += amount_ml
        found = True
        if location:
            location_totals[location] += amount_ml
            location_found[location] = True
        else:
            unlabeled_amounts.append(amount_ml)

    if not found:
        return result

    if system == "Tower":
        if unlabeled_amounts:
            location_totals["towers"] += sum(unlabeled_amounts)
            location_found["towers"] = True
    elif system == "A-shape + Gutters":
        remaining_locations = ["nft", "gutters"]
        for amount_ml in unlabeled_amounts:
            target = None
            for location in remaining_locations:
                if not location_found[location]:
                    target = location
                    break
            if target is None:
                target = "nft"
            location_totals[target] += amount_ml
            location_found[target] = True
    result["ph_down_ml"] = total

    for location, value in location_totals.items():
        if location_found[location]:
            result[f"ph_down_{location}_ml"] = value
    return result


def standardize_leak_flag(text):
    text = clean_text(text)
    if pd.isna(text):
        return "Unknown"
    lowered = text.lower()
    if lowered == "no" or "no more leaking" in lowered:
        return "No"
    if any(token in lowered for token in ["yes", "leak", "leaking", "major", "minor", "so much"]):
        return "Yes"
    return "Unknown"


def standardize_leak_severity(text):
    text = clean_text(text)
    if pd.isna(text):
        return "Unknown"
    lowered = text.lower()
    if lowered == "no" or "no more leaking" in lowered:
        return "No Leak"
    if any(token in lowered for token in ["major", "alot", "a lot", "so much"]):
        return "Major"
    if any(token in lowered for token in ["minor", "a little", "a bit", "abit"]):
        return "Minor"
    if "leak" in lowered or "yes" in lowered:
        return "Unspecified"
    return "Unknown"


def extract_leak_locations(*texts):
    patterns = [
        (r"\br1\b", "R1"),
        (r"a shape|a-shape", "A-shape"),
        (r"gutters?", "Gutters"),
        (r"\bnft\b", "NFT"),
        (r"temporary tank|temp tank", "Temporary tank"),
        (r"cylinders?", "Cylinders"),
    ]
    found = []
    for text in texts:
        text = clean_text(text)
        if pd.isna(text):
            continue
        lowered = text.lower()
        for pattern, label in patterns:
            if re.search(pattern, lowered) and label not in found:
                found.append(label)
    if found:
        return "; ".join(found)
    return "Not Reported"


def standardize_stage(text):
    text = clean_text(text)
    if pd.isna(text):
        return "Unknown"
    lowered = text.lower()
    if "transplant" in lowered:
        return "Transplanted"
    if "seeds and" in lowered and "seedling" in lowered:
        return "Mixed"
    if "seedlings" in lowered:
        return "Seedling"
    if "seeds" in lowered:
        return "Seed"
    return "Unknown"


def parse_age_days(text):
    text = clean_text(text)
    if pd.isna(text):
        return np.nan
    lowered = text.lower()
    day_range = re.search(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*days?", lowered)
    if day_range:
        return (float(day_range.group(1)) + float(day_range.group(2))) / 2
    week_range = re.search(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*weeks?", lowered)
    if week_range:
        return ((float(week_range.group(1)) + float(week_range.group(2))) / 2) * 7
    day_match = re.search(r"(\d+(?:\.\d+)?)\s*days?", lowered)
    if day_match:
        return float(day_match.group(1))
    week_match = re.search(r"(\d+(?:\.\d+)?)\s*weeks?", lowered)
    if week_match:
        return float(week_match.group(1)) * 7
    if lowered == "0":
        return 0.0
    return np.nan


def parse_tower_planted_count(text):
    text = clean_text(text)
    if pd.isna(text):
        return np.nan
    lowered = text.lower()
    total_towers = 24
    if "harvested everything" in lowered or "end of cycle" in lowered:
        return 0.0
    if "all except" in lowered:
        units_missing = 0
        towers_missing = 0
        unit_match = re.search(r"all except\s+(\d+)\s+unit", lowered)
        if unit_match:
            units_missing = int(unit_match.group(1))
        elif "all except 1 unit" in lowered:
            units_missing = 1
        tower_match = re.search(r"and\s+(\d+)\s+tower", lowered)
        if tower_match:
            towers_missing = int(tower_match.group(1))
        elif "all except 1 tower" in lowered:
            towers_missing = 1
        return float(total_towers - units_missing * 6 - towers_missing)

    units_empty_match = re.search(r"(\d+)\s+units?\s+empty", lowered)
    towers_empty_match = re.search(r"(\d+)\s+towers?\s+empty", lowered)
    if units_empty_match or towers_empty_match:
        units_empty = int(units_empty_match.group(1)) if units_empty_match else 0
        towers_empty = int(towers_empty_match.group(1)) if towers_empty_match else 0
        return float(total_towers - units_empty * 6 - towers_empty)
    return np.nan


def parse_general_plant_count(text):
    text = clean_text(text)
    if pd.isna(text):
        return np.nan
    lowered = text.lower()
    pepper_match = re.search(r"\((\d+)\s+peppers?\)", lowered)
    if pepper_match:
        return float(pepper_match.group(1))
    plants_match = re.search(r"(\d+)\s+plants?\b", lowered)
    if plants_match:
        return float(plants_match.group(1))
    if "seeding tray" in lowered:
        numbers = [int(x) for x in re.findall(r"\d+", lowered)]
        return float(numbers[0]) if numbers else np.nan

    numbers = []
    for match in re.finditer(r"(\d+)", lowered):
        tail = lowered[match.end() : match.end() + 12]
        if "day" in tail:
            continue
        numbers.append(int(match.group(1)))
    if numbers:
        return float(sum(numbers))
    return np.nan


def extract_crops(*texts):
    found = []
    crop_map = [
        ("pepper", "Pepper"),
        ("basil", "Basil"),
        ("arugula", "Arugula"),
        ("lettuce", "Lettuce"),
        ("kale", "Kale"),
    ]
    for text in texts:
        text = clean_text(text)
        if pd.isna(text):
            continue
        lowered = text.lower()
        for token, label in crop_map:
            if token in lowered and label not in found:
                found.append(label)
    return ", ".join(found) if found else "Unknown"


def categorize_problem_notes(text):
    text = clean_text(text)
    if pd.isna(text):
        return "No Issue Recorded"
    lowered = text.lower()
    categories = []
    rules = [
        (["light"], "Lights"),
        (["electricity"], "Electricity"),
        (["pump"], "Pump"),
        (["sensor", "floater"], "Sensor/Floater"),
        (["spill", "spilled", "overflow"], "Spill/Overflow"),
        (["evaporation", "door was open"], "Evaporation"),
        (["water stress"], "Water Stress"),
        (["whitefl", "weed"], "Pests/Weeds"),
        (["harvest"], "Harvest"),
        (["installation", "ozonation"], "Maintenance"),
        ([" ec ", "ec 1700", "ec 1500"], "EC/Nutrient"),
    ]
    padded = f" {lowered} "
    for tokens, label in rules:
        if any(token in padded or token in lowered for token in tokens):
            categories.append(label)
    return "; ".join(dict.fromkeys(categories)) if categories else "Other"


def contains_any(text, tokens):
    text = clean_text(text)
    if pd.isna(text):
        return False
    lowered = text.lower()
    return any(token in lowered for token in tokens)



## Step 3. Main cleaning and feature engineering

This is the core transformation step.

It:

- maps each sheet to the correct analytical system
- removes duplicates
- converts dates and times
- parses water, nutrient, pH-down, and duration values
- builds leak, crop, plant-count, and issue-category fields
- creates a unified `water_use_l` column for business comparison


In [9]:

sheet_to_system = {
    "Towers": ("Tower", "Hydroponic"),
    "A shape and Gutters": ("A-shape + Gutters", "Hydroponic"),
    "Conventional": ("Conventional", "Soil"),
}

clean_df = raw_df.copy()
clean_df["system"] = clean_df["source_sheet"].map(lambda x: sheet_to_system.get(x, ("Unknown", "Unknown"))[0])
clean_df["system_type"] = clean_df["source_sheet"].map(lambda x: sheet_to_system.get(x, ("Unknown", "Unknown"))[1])

before_dedup = len(clean_df)
clean_df = clean_df.drop_duplicates().copy()
duplicates_removed = before_dedup - len(clean_df)

clean_df["observation_date"] = clean_df["Date"].apply(excel_serial_to_date)
clean_df["observation_time"] = clean_df["Time"].apply(excel_fraction_to_time)
clean_df = clean_df.sort_values(["system", "observation_date", "source_row_number"]).reset_index(drop=True)

clean_df["observation_time"] = clean_df.groupby("system")["observation_time"].ffill()

clean_df["observation_timestamp"] = pd.to_datetime(
    clean_df["observation_date"].astype("string") + " " + clean_df["observation_time"].fillna("00:00:00"),
    errors="coerce",
)

water_in_return_split = clean_df["water in return"].apply(split_value_and_comment)
clean_df["water_in_return_l"] = water_in_return_split.apply(lambda x: x[0])
clean_df["water_in_return_comment"] = water_in_return_split.apply(lambda x: x[1])

return_now_split = clean_df["return now"].apply(split_value_and_comment)
clean_df["return_now_l"] = return_now_split.apply(lambda x: x[0])
clean_df["return_now_comment"] = return_now_split.apply(lambda x: x[1])

water_consumed_split = clean_df["how much consumed"].apply(split_value_and_comment)
clean_df["water_consumed_l"] = water_consumed_split.apply(lambda x: x[0])
clean_df["water_consumed_comment"] = water_consumed_split.apply(lambda x: x[1])
clean_df["watered_amount_l"] = clean_df["watered amount"].apply(parse_watered_amount_liters)

parsed_duration = clean_df["mins added"].apply(parse_duration_minutes)
clean_df["water_addition_duration_min"] = parsed_duration.apply(lambda x: x[0])
clean_df["water_addition_method"] = parsed_duration.apply(lambda x: x[1])
clean_df["manual_water_addition"] = parsed_duration.apply(lambda x: x[2])

parsed_nutrients = pd.DataFrame(
    clean_df.apply(lambda row: parse_nutrient_solution(row.get("nutrient solution"), row.get("system")), axis=1).tolist(),
    index=clean_df.index,
)
for col in parsed_nutrients.columns:
    clean_df[col] = parsed_nutrients[col]
clean_df["nutrient_entry_type"] = clean_df["nutrient solution"].apply(classify_nutrient_entry)

parsed_ph = pd.DataFrame(
    clean_df.apply(lambda row: parse_ph_down_by_location(row.get("pH down"), row.get("system")), axis=1).tolist(),
    index=clean_df.index,
)
for col in parsed_ph.columns:
    clean_df[col] = parsed_ph[col]
clean_df["ph_adjustment_flag"] = clean_df["ph_down_ml"].notna()

clean_df["leak_flag"] = clean_df["leak or no"].apply(standardize_leak_flag)
clean_df["leak_severity"] = clean_df["leak or no"].apply(standardize_leak_severity)
clean_df["problem_notes"] = clean_df["problems?"].combine_first(clean_df["problems faced"])
clean_df["leak_locations"] = clean_df.apply(
    lambda row: extract_leak_locations(row.get("leak or no"), row.get("problem_notes")),
    axis=1,
)

clean_df["plant_name"] = clean_df["plant"].fillna(pd.NA).astype("string").str.title()
clean_df["growth_stage"] = clean_df["seed or seedling"].apply(standardize_stage)
clean_df["age_days"] = clean_df["age"].apply(parse_age_days)
clean_df["planting_notes"] = clean_df["how many planted"].combine_first(clean_df["how many plants planted?"])
clean_df["crop_types"] = clean_df.apply(
    lambda row: extract_crops(row.get("plant"), row.get("planting_notes"), row.get("problem_notes")),
    axis=1,
)

clean_df["plant_count"] = np.nan
tower_mask = clean_df["system"] == "Tower"
ag_mask = clean_df["system"] == "A-shape + Gutters"
conv_mask = clean_df["system"] == "Conventional"
clean_df.loc[tower_mask, "plant_count"] = clean_df.loc[tower_mask, "how many planted"].apply(parse_tower_planted_count)
clean_df.loc[ag_mask, "plant_count"] = clean_df.loc[ag_mask, "how many planted"].apply(parse_general_plant_count)
clean_df.loc[conv_mask, "plant_count"] = clean_df.loc[conv_mask, "how many plants planted?"].apply(parse_general_plant_count)

clean_df["weekend_or_aggregate_flag"] = (
    clean_df["how much consumed"].apply(lambda x: contains_any(x, ["weekend", "holiday", "2 days", "1 cycle", "4 hours"]))
    | clean_df["watered amount"].apply(lambda x: contains_any(x, ["weekend"]))
)
clean_df["estimated_value_flag"] = (
    clean_df["how much consumed"].apply(lambda x: contains_any(x, ["?", "assuming"]))
    | clean_df["watered amount"].apply(lambda x: contains_any(x, ["?", "ig"]))
)
clean_df["harvest_related_flag"] = (
    clean_df["problem_notes"].apply(lambda x: contains_any(x, ["harvest"]))
    | clean_df["planting_notes"].apply(lambda x: contains_any(x, ["harvest"]))
)
clean_df["end_of_cycle_flag"] = (
    clean_df["mins added"].apply(lambda x: contains_any(x, ["end of cycle"]))
    | clean_df["planting_notes"].apply(lambda x: contains_any(x, ["end of cycle"]))
)
clean_df["problem_categories"] = clean_df["problem_notes"].apply(categorize_problem_notes)
clean_df["issue_flag"] = np.where(
    (clean_df["leak_flag"] == "Yes") | clean_df["problem_notes"].notna(),
    "Yes",
    "No",
)

clean_df["water_use_l"] = np.where(
    clean_df["system"] == "Conventional",
    clean_df["watered_amount_l"],
    clean_df["water_consumed_l"],
)
clean_df["water_use_basis"] = np.where(
    clean_df["system"] == "Conventional",
    "Applied watering",
    "Recorded consumption",
)

clean_df["nutrient_addition_flag"] = clean_df["nutrient_total_ml"].notna() | (clean_df["nutrient_entry_type"] != "No Addition")
clean_df["manual_intervention_flag"] = (
    clean_df["manual_water_addition"]
    | clean_df["ph_adjustment_flag"]
    | clean_df["nutrient_addition_flag"]
    | (clean_df["issue_flag"] == "Yes")
)

numeric_cols = [
    "water_in_return_l",
    "return_now_l",
    "water_consumed_l",
    "watered_amount_l",
    "water_addition_duration_min",
    "nutrient_a_ml",
    "nutrient_b_ml",
    "nutrient_total_ml",
    "nutrient_a_nft_ml",
    "nutrient_b_nft_ml",
    "nutrient_a_gutters_ml",
    "nutrient_b_gutters_ml",
    "nutrient_a_towers_ml",
    "nutrient_b_towers_ml",
    "ph_down_ml",
    "ph_down_nft_ml",
    "ph_down_gutters_ml",
    "ph_down_towers_ml",
    "age_days",
    "plant_count",
    "water_use_l",
]
for col in numeric_cols:
    clean_df.loc[clean_df[col] < 0, col] = np.nan

clean_df = clean_df[clean_df["system"].notna() & clean_df["observation_date"].notna()].copy()

print("Rows after structural cleaning:", len(clean_df))
print("Duplicates removed:", duplicates_removed)
clean_df.head()


Rows after structural cleaning: 276
Duplicates removed: 0


,source_sheet,source_row_number,Date,Time,water in return,mins added,return now,nutrient solution,pH down,how much consumed,...,weekend_or_aggregate_flag,estimated_value_flag,harvest_related_flag,end_of_cycle_flag,problem_categories,issue_flag,water_use_l,water_use_basis,nutrient_addition_flag,manual_intervention_flag
0,A shape and Gutters,2,45883,0.66666666666666663,<NA>,<NA>,240L,2L A 2L B NFT and 200 A 200 B gutters,<NA>,<NA>,...,False,False,False,False,No Issue Recorded,No,NaN,Recorded consumption,True,True
1,A shape and Gutters,3,45886,0.56736111111111109,60L gutters supply almost empty,5 mins,180L,500 A 500 B NFT 300 A 300 B gutters,<NA>,180L just return tank (weekend),...,True,False,False,False,Lights,Yes,180.0,Recorded consumption,True,True
2,A shape and Gutters,4,45887,0.51041666666666663,70L gutters supply missing 2-3L,3 mins,126L,<NA>,<NA>,113L return+gutters supply,...,False,False,False,False,Lights,Yes,113.0,Recorded consumption,False,True
3,A shape and Gutters,5,45888,0.48402777777777778,80L,manually,190L,<NA>,<NA>,46L,...,False,False,False,False,Lights,Yes,46.0,Recorded consumption,False,True
4,A shape and Gutters,6,45889,0.49305555555555558,124L,manually,197L,<NA>,<NA>,66L,...,False,False,False,False,Lights,Yes,66.0,Recorded consumption,False,True



## Step 4. Missing-data policy and row filtering

The missing-data policy is applied carefully.

### Rules used

- **Small / irrelevant**: drop rows that have no operational measurement and no meaningful event signal
- **Numeric**: median-fill only the low-risk descriptive field `age_days`, and only for `Conventional`
- **Time-based**: forward fill `observation_time` within each system
- **Categorical**: fill support fields with `"Unknown"`
- **Important measured data**: leave missing instead of faking values

This keeps the dataset cleaner without pretending the source data was more complete than it really was.


In [10]:

clean_df["observation_time"] = clean_df.groupby("system")["observation_time"].ffill()
clean_df["observation_timestamp"] = pd.to_datetime(
    clean_df["observation_date"].astype("string") + " " + clean_df["observation_time"].fillna("00:00:00"),
    errors="coerce",
)

clean_df["age_days_imputed"] = False
conventional_mask = clean_df["system"] == "Conventional"
conventional_age_median = clean_df.loc[conventional_mask, "age_days"].median(skipna=True)
missing_conventional_age = conventional_mask & clean_df["age_days"].isna()
clean_df.loc[missing_conventional_age, "age_days"] = conventional_age_median
clean_df.loc[missing_conventional_age, "age_days_imputed"] = True

clean_df["nutrient_entry_type"] = clean_df["nutrient_entry_type"].fillna("No Addition")
clean_df["peristaltic_pump"] = clean_df["peristaltic_pump"].fillna("No")

categorical_fill_cols = [
    "leak_flag",
    "leak_severity",
    "leak_locations",
    "plant_name",
    "growth_stage",
    "planting_notes",
    "problem_notes",
    "problem_categories",
    "crop_types",
]
for col in categorical_fill_cols:
    clean_df[col] = clean_df[col].fillna("Unknown")

important_numeric_cols = [
    "water_in_return_l",
    "return_now_l",
    "water_consumed_l",
    "watered_amount_l",
    "water_addition_duration_min",
    "nutrient_a_ml",
    "nutrient_b_ml",
    "nutrient_total_ml",
    "ph_down_ml",
    "water_use_l",
]
clean_df["core_measurement_missing"] = clean_df[important_numeric_cols].isna().all(axis=1)

clean_df["drop_irrelevant_sparse_row"] = (
    clean_df["core_measurement_missing"]
    & (clean_df["issue_flag"] == "No")
    & (~clean_df["harvest_related_flag"])
)
rows_before_drop = len(clean_df)
clean_df = clean_df.loc[~clean_df["drop_irrelevant_sparse_row"]].copy()
rows_dropped_as_irrelevant = rows_before_drop - len(clean_df)

clean_df["data_quality_status"] = np.select(
    [
        clean_df["estimated_value_flag"],
        clean_df["weekend_or_aggregate_flag"],
        clean_df["core_measurement_missing"],
    ],
    [
        "Estimated",
        "Aggregate",
        "Event Only",
    ],
    default="Usable",
)

clean_df["analysis_ready_water_use_flag"] = (
    clean_df["water_use_l"].notna()
    & (~clean_df["estimated_value_flag"])
    & (~clean_df["core_measurement_missing"])
)

clean_df["sanity_warning_flag"] = (
    ((clean_df["system"] == "Tower") & (clean_df["plant_count"] > 24))
    | (clean_df["water_use_l"] < 0)
    | (clean_df["water_in_return_l"] < 0)
    | (clean_df["return_now_l"] < 0)
)

final_columns = [
    "source_sheet",
    "source_row_number",
    "system",
    "system_type",
    "observation_date",
    "observation_time",
    "observation_timestamp",
    "water in return",
    "water_in_return_l",
    "water_in_return_comment",
    "mins added",
    "water_addition_method",
    "water_addition_duration_min",
    "manual_water_addition",
    "return now",
    "return_now_l",
    "return_now_comment",
    "nutrient solution",
    "nutrient_entry_type",
    "nutrient_solution_note",
    "peristaltic_pump",
    "nutrient_a_ml",
    "nutrient_b_ml",
    "nutrient_total_ml",
    "nutrient_a_nft_ml",
    "nutrient_b_nft_ml",
    "nutrient_a_gutters_ml",
    "nutrient_b_gutters_ml",
    "nutrient_a_towers_ml",
    "nutrient_b_towers_ml",
    "pH down",
    "ph_down_ml",
    "ph_down_nft_ml",
    "ph_down_gutters_ml",
    "ph_down_towers_ml",
    "ph_adjustment_flag",
    "how much consumed",
    "water_consumed_l",
    "water_consumed_comment",
    "watered amount",
    "watered_amount_l",
    "water_use_l",
    "water_use_basis",
    "leak or no",
    "leak_flag",
    "leak_severity",
    "leak_locations",
    "plant",
    "plant_name",
    "seed or seedling",
    "growth_stage",
    "age",
    "age_days",
    "age_days_imputed",
    "how many planted",
    "how many plants planted?",
    "planting_notes",
    "plant_count",
    "crop_types",
    "problems?",
    "problems faced",
    "problem_notes",
    "problem_categories",
    "issue_flag",
    "nutrient_addition_flag",
    "manual_intervention_flag",
    "weekend_or_aggregate_flag",
    "estimated_value_flag",
    "harvest_related_flag",
    "end_of_cycle_flag",
    "core_measurement_missing",
    "drop_irrelevant_sparse_row",
    "analysis_ready_water_use_flag",
    "sanity_warning_flag",
    "data_quality_status",
]
final_df = clean_df[final_columns].copy()
final_df.head()


,source_sheet,source_row_number,system,system_type,observation_date,observation_time,observation_timestamp,water in return,water_in_return_l,water_in_return_comment,...,manual_intervention_flag,weekend_or_aggregate_flag,estimated_value_flag,harvest_related_flag,end_of_cycle_flag,core_measurement_missing,drop_irrelevant_sparse_row,analysis_ready_water_use_flag,sanity_warning_flag,data_quality_status
0,A shape and Gutters,2,A-shape + Gutters,Hydroponic,2025-08-14,16:00:00,2025-08-14 16:00:00,<NA>,NaN,<NA>,...,True,False,False,False,False,False,False,False,False,Usable
1,A shape and Gutters,3,A-shape + Gutters,Hydroponic,2025-08-17,13:37:00,2025-08-17 13:37:00,60L gutters supply almost empty,60.0,gutters supply almost empty,...,True,True,False,False,False,False,False,True,False,Aggregate
2,A shape and Gutters,4,A-shape + Gutters,Hydroponic,2025-08-18,12:15:00,2025-08-18 12:15:00,70L gutters supply missing 2-3L,70.0,gutters supply missing 2-3L,...,True,False,False,False,False,False,False,True,False,Usable
3,A shape and Gutters,5,A-shape + Gutters,Hydroponic,2025-08-19,11:37:00,2025-08-19 11:37:00,80L,80.0,<NA>,...,True,False,False,False,False,False,False,True,False,Usable
4,A shape and Gutters,6,A-shape + Gutters,Hydroponic,2025-08-20,11:50:00,2025-08-20 11:50:00,124L,124.0,<NA>,...,True,False,False,False,False,False,False,True,False,Usable



## Step 5. Sanity checks and export

Final checks confirm that the cleaned table is internally consistent before export.

Checks include:

- duplicate count
- missing dates
- rows dropped as irrelevant
- time and age imputations
- sanity warning flags
- system counts and quality-status counts


In [12]:

duplicate_count = final_df.duplicated().sum()
missing_date_count = final_df["observation_date"].isna().sum()

print("Final rows:", len(final_df))
print("Duplicate rows remaining:", int(duplicate_count))
print("Missing observation dates:", int(missing_date_count))
print("Rows dropped as irrelevant sparse rows:", int(rows_dropped_as_irrelevant))
print("Median-imputed age values:", int(final_df["age_days_imputed"].sum()))
print("Sanity warnings:", int(final_df["sanity_warning_flag"].sum()))
print()
print("Rows by system:")
print(final_df["system"].value_counts())
print()
print("Data quality status:")
print(final_df["data_quality_status"].value_counts())
print()
print("Analysis-ready water-use rows:")
print(final_df["analysis_ready_water_use_flag"].value_counts())

final_df.to_csv(OUTPUT_FILE, index=False)
print()
print("Cleaned CSV written to:", OUTPUT_FILE.resolve())


Final rows: 272
Duplicate rows remaining: 0
Missing observation dates: 0
Rows dropped as irrelevant sparse rows: 4
Median-imputed age values: 2
Sanity warnings: 0

Rows by system:
system
Conventional         113
A-shape + Gutters     97
Tower                 62
Name: count, dtype: int64

Data quality status:
data_quality_status
Usable        235
Aggregate      29
Estimated       6
Event Only      2
Name: count, dtype: int64

Analysis-ready water-use rows:
analysis_ready_water_use_flag
True     241
False     31
Name: count, dtype: int64

Cleaned CSV written to: /content/greenhouse_systems_cleaned.csv
